In [1]:
!pip install sentence-transformers scikit-learn python-dotenv

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

WIKI = Path(os.environ["WIKI_PATH"])

notes = []

for file in WIKI.rglob("*.md"):
    text = file.read_text(encoding="utf-8")
    notes.append({"path": str(file), "text": text})

print(f"Loaded {len(notes)} notes")
print(notes[0]["path"])
print(notes[0]["text"][:300])

Loaded 91 notes
/Users/sabimantock/Library/Mobile Documents/iCloud~md~obsidian/Documents/My Brain/wiki/index.md
# Wiki Index

Content catalog — updated after every ingest.

---

## Sources

- [[wiki/sources/ai-2027]] — Research-backed scenario forecasting AGI by 2027; covers timelines, alignment failure, US-China race, and intelligence explosion (1 source)
- [[wiki/sources/ai-in-engineering-education-review]]


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # the thing that draws the meaning map

texts = [n["text"] for n in notes]                # just the text of each note
embeddings = model.encode(texts, show_progress_bar=True)

print(embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(91, 384)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

question = "what is embedding"
q_emb = model.encode([question])                    # the question becomes a dot too

scores = cosine_similarity(q_emb, embeddings)[0]     # how close it is to every note
top = np.argsort(scores)[::-1][:3]                   # the 3 nearest dots

for i in top:
    print(round(float(scores[i]), 3), notes[i]["path"].split("/")[-1])

0.334 synapse-rag.md
0.32 nlp-embeddings-flashcards.md
0.304 nlp-text-vectorization.md


In [5]:
!pip install anthropic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# 1. glue the top retrieved notes together into one block of context
context = "\n\n---\n\n".join(notes[i]["text"] for i in top)

# 2. build the prompt: give it the notes, then the question
prompt = f"""Answer the question using ONLY the notes below.
If the answer isn't in the notes, say you don't know.

NOTES:
{context}

QUESTION: {question}"""

# 3. send it to the LLM and print the grounded answer
resp = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}],
)
print(resp.content[0].text)

## What is an Embedding?

Based on the notes, an embedding is a word's (or sentence's) **spot on a map of meaning**, written as **numbers (coordinates)**.

### Key properties:
- Words with **similar meaning are placed near each other**; unrelated words are placed **far apart**
  - e.g. "cash" sits next to "money"; "banana" sits over by "apple"
- The map has **hundreds of directions** (dimensions), not just two — which is why an embedding is a **long list of numbers**
  - The `all-MiniLM-L6-v2` model used in Synapse produces **384-dimensional** vectors
- The core rule: **near = similar meaning, far = different meaning**

### How placement is learned:
The model learns by reading millions of sentences and noticing which words are surrounded by the **same neighbor words**. For example, "money" and "cash" both appear near words like *borrow, pay, bank, withdraw* — so they end up as neighbors on the map. This is called **"the company they keep."**

### Not just words:
A whole **sentence, par